In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

target_model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
draft_model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

target_model = AutoModelForCausalLM.from_pretrained(target_model_id)
tokenizer = AutoTokenizer.from_pretrained(target_model_id)

draft_model = AutoModelForCausalLM.from_pretrained(draft_model_id)

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
from transformers import TextIteratorStreamer
import torch, threading, time

target_model.generation_config.max_length   = None
target_model.generation_config.min_length   = None      # ← this is the culprit
target_model.generation_config.min_new_tokens = None


streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

inputs = tokenizer("The greatest", return_tensors="pt")

thread = threading.Thread(
    target=target_model.generate,
    kwargs=dict(
        **inputs,
        assistant_model=draft_model,
        max_new_tokens=200,
        streamer=streamer,
        min_length=0,
        min_new_tokens=None,
        do_sample=False,
        use_cache=True,
    )
)

token_count = 0
start = time.time()

thread.start()

print("Response:\n" + "-"*40)
for token in streamer:
    print(token, end="", flush=True)
    token_count += len(tokenizer.encode(token, add_special_tokens=False))

thread.join()

elapsed = time.time() - start
print(f"\n" + "-"*40)
print(f"Tokens generated : {token_count}")
print(f"Time elapsed     : {elapsed:.2f}s")
print(f"Throughput       : {token_count/elapsed:.1f} tokens/sec")

Response:
----------------------------------------
 common divisor (GCD) of two numbers is the largest number that divides both numbers without leaving a remainder. In other words, it's the largest number that can be found by dividing both numbers and leaving no remainder.

For example, the GCD of 48 and 18 is 6, because 6 is the largest number that divides both 48 and 18 without leaving a remainder.

The GCD is an important concept in number theory, as it has many applications in mathematics, such as simplifying fractions, solving equations, and finding the least common multiple (LCM) of two numbers.

In the context of the original problem, the GCD of 120 and 18 is 6, because 6 is the largest number that divides both 120 and 18 without leaving a remainder.
----------------------------------------
Tokens generated : 213
Time elapsed     : 20.38s
Throughput       : 10.5 tokens/sec


In [5]:

def get_tokens(target_model, draft_model, tokenizer, prompt):
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    inputs = tokenizer(prompt, return_tensors="pt")

    thread = threading.Thread(
        target=target_model.generate,
        kwargs=dict(
            **inputs,
            assistant_model=draft_model,
            max_new_tokens=200,
            streamer=streamer,
            min_length=0,
            min_new_tokens=None,
            do_sample=False,
            use_cache=True,
        )
    )

    for token in streamer:
        # print(token, end="", flush=True)
        token_count += len(tokenizer.encode(token, add_special_tokens=False))
        yield token, token_count
    thread.join()

In [7]:
for i,j in get_tokens(target_model, draft_model, tokenizer, "The greatest"):
    print(i, j)

KeyboardInterrupt: 